# ANN Implementation with pytorch on Fashion MNIST

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
# Check the ability of GPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataloaders

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="../../datasets/",
    train = True,
    download = True,
    transform=transform,
)

test_data = datasets.FashionMNIST(
    root="../../datasets/",
    train=False,
    download=True,
    transform=transform
)


train_loader = DataLoader(train_data, batch_size=64*8, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False, pin_memory=True)


In [ ]:
# Preprocess data
feature, label = train_data[0]
print(f"Feature: {feature}")
print(f"Label: {label}")

In [ ]:
# Model Architecture
# L1: Input of 784
# L2: Hidden Layer 128, ReLU
# L3: Hidden Layer 64, ReLU
# L4: Output Layer 10, Softmax
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(), # Flatten the input image
            nn.Linear(28 * 28, 128), # in 784, out 128
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
            # nn.Softmax() # Explicit softmax definition not needed. By default added with softmax
        )
    def forward(self, x):
        
        return self.model(x)
        

In [ ]:
# Learning rate and epochs
epochs = 15
learning_rate = 0.1

In [ ]:

model = ANN()
model.to(device=device)

# Loss function
criterion = nn.CrossEntropyLoss() # Not binary cross entropy

# Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [ ]:
# Training loop
for epoch in range(epochs):
    epoch_loss = 0
    for batch_features, batch_labels in train_loader                        :
        
        # Move batch_features and batch_labels to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        
        # Calculate Loss
        loss = criterion(outputs, batch_labels)
        epoch_loss += loss.item()
        # Clear gradients
        optimizer.zero_grad()
        
        # Backward Pass
        loss.backward()
        
        # Update Gradients
        optimizer.step()
        
    print(f"Epoch :{epoch + 1}, Loss: {epoch_loss/len(train_data)}")

In [ ]:
# Evaluation
test_features, test_labels = next(iter(test_loader))
test_features, test_labels = test_features.to(device), test_labels.to(device)


# Prepare model for evaluation mode
model.eval()

with torch.no_grad():
    logits = model(test_features)
    
    # Convert to numpy for using scikit learn's methods
    preds = logits.argmax(dim=1).cpu().numpy()
    labels = test_labels.cpu().numpy()
    
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')
    f1 = f1_score(labels, preds, average='macro')
    
print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1 Score: {f1}")
    